# Rule: **build_biomass_potentials**


**Description**

This rules computes biogas and solid biomass potentials for each clustered model region using data from JRC ENSPRESO.

The additional configuration parameters associated with this rule are defined under the **biomass** section of the config file:  
- biomass.year  
- biomass.scenario  
- biomass.classes  
- biomass.share_unsustainable_use_retained  
- biomass.share_sustainable_potential_available


**Outputs**

- resources/{prefix}/{name}/`biomass_potentials_all_{clusters}_{horizon}.csv`
- resources/{prefix}/{name}/`biomass_potentials_s_{clusters}_{horizon}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'
opts = ''
sector_opts = ''
horizon = ''

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = 'ES'
country_name = 'Spain'

In [ ]:
##### Imports
import pandas as pd
import geopandas as gpd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import os 
import sys
from matplotlib.colors import Normalize

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Region files
region_tag = f"base_s_{clusters}"
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)

##### Set options
pd.set_option("display.max_columns", None)

## `biomass_potentials_all_{clusters}_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"biomass_potentials_all_{clusters}_{horizon}.csv"

biomass_potentials_all = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

biomass_potentials_all.head()

## `biomass_potentials_s_{clusters}_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"biomass_potentials_s_{clusters}_{horizon}.csv"

biomass_potentials = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

print('Biomass energy potentials in MWh/a')
biomass_potentials.head()

What is the spatial distribution of bioenergy potential across the country?

In [ ]:
#################### Parameters 

### Biomass components to plot. Choose any subset, in desired order.
biomass_variables = [
    'solid biomass',
    'biogas',    
    #'municipal solid waste',
    #'unsustainable solid biomass',
    #'unsustainable biogas',
    #'unsustainable bioliquids',
    #'not included',
]


### Plotting parameters
font_size = 15



#################### Plot

# Convert from MWh/a to TWh/a
biomass_twh = biomass_potentials.copy()
for var in biomass_variables:
    biomass_twh[var] = biomass_twh[var] / 1e6

# Merge with regions
gdf = gdf_regions_onshore.merge(
    biomass_twh,
    left_on="name",
    right_on=biomass_twh.columns[0],
    how="left"
)

# Common color scale
vmin = gdf[biomass_variables].min().min()
vmax = gdf[biomass_variables].max().max()
norm = Normalize(vmin=vmin, vmax=vmax)

# Layout: 2 columns, rows determined by the number of selected variables
n_vars = len(biomass_variables)
n_cols = min(2, n_vars)
n_rows = int(np.ceil(n_vars / n_cols))

fig_size = [6 * n_cols, 5 * n_rows]
crs = ccrs.PlateCarree()

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=fig_size,
    constrained_layout=True,
    subplot_kw={'projection': crs},
)

# Flatten axes for easy iteration (works for 1, 2 or more subplots)
axes_flat = np.atleast_1d(axes).flatten()

for ax, var in zip(axes_flat, biomass_variables):
    gdf.plot(
        column=var,
        cmap="YlGn",
        norm=norm,
        linewidth=0.8,
        ax=ax,
        edgecolor="black",
        legend=False
    )
    total = gdf[var].sum()
    ax.set_title(f"{var.capitalize()} (total: {total:.1f} TWh/a)", fontsize=font_size)


    ### Add map features
    xp.map_add_features(ax, params['map_add_features'])

# Hide any unused subplots
for ax in axes_flat[n_vars:]:
    ax.set_visible(False)

# Shared colorbar (aligned with the visible subplots)
sm = plt.cm.ScalarMappable(cmap="YlGn", norm=norm)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes_flat[:n_vars].tolist(),
    location="right",
    shrink=0.8
)
cbar.set_label("TWh/year", fontsize=font_size)
cbar.ax.tick_params(labelsize=font_size)

fig.suptitle(
    "Biomass energy potentials by model region in Spain",
    fontsize=font_size*1.2
)